### 导入必要的库

In [1]:
import torch

In [2]:
  print(torch.__version__)

2.0.0+cpu


In [3]:
import numpy as np

In [4]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

### 生成示例数据

In [5]:
# 设置随机种子以确保结果可复现
torch.manual_seed(0)

# 生成100个样本，每个样本有5个特征
X_tensor = torch.randn(100, 5)

# 生成100个标签，取值范围为0到2（共3个类别）
y_tensor = torch.randint(0, 3, (100,))

In [6]:
# 创建TensorDataset
dataset = TensorDataset(X_tensor, y_tensor)

# 创建DataLoader
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

### 定义神经网络

In [7]:
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

### 初始化模型、损失函数和优化器

In [8]:
input_size = 5  # 输入特征的数量
hidden_size = 64  # 隐藏层的大小
output_size = 3  # 输出类别的数量

model = SimpleNN(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

### 训练模型

In [9]:
num_epochs = 10

for epoch in range(num_epochs):
    for inputs, targets in dataloader:
        # 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [1/10], Loss: 1.0184
Epoch [2/10], Loss: 1.1042
Epoch [3/10], Loss: 1.1467
Epoch [4/10], Loss: 1.0798
Epoch [5/10], Loss: 1.1109
Epoch [6/10], Loss: 1.1323
Epoch [7/10], Loss: 1.0685
Epoch [8/10], Loss: 1.0770
Epoch [9/10], Loss: 1.1066
Epoch [10/10], Loss: 1.1287


In [10]:
### 评估模型

# 假设 X_test 和 y_test 是测试集的特征和标签
X_test_tensor = torch.randn(20, 5)  # 生成20个测试样本
y_test_tensor = torch.randint(0, 3, (20,))  # 生成20个标签

# 禁用梯度计算
with torch.no_grad():
    model.eval()  # 设置模型为评估模式
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f'Accuracy: {accuracy * 100:.2f}%')


Accuracy: 20.00%


### 比较使用和不使用pytorch

In [11]:
## 使用pytorch
import torch
import torch.nn as nn
import torch.optim as optim

# 定义模型
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)  # 输入和输出都是1维

    def forward(self, x):
        return self.linear(x)

# 生成示例数据
X = torch.randn(100, 1)
y = 3 * X + 2 + torch.randn(100, 1) * 0.1  # y = 3x + 2 + 噪声

# 初始化模型、损失函数和优化器
model = LinearRegressionModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 训练模型
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')


Epoch [10/100], Loss: 10.0745
Epoch [20/100], Loss: 6.5318
Epoch [30/100], Loss: 4.2371
Epoch [40/100], Loss: 2.7505
Epoch [50/100], Loss: 1.7873
Epoch [60/100], Loss: 1.1630
Epoch [70/100], Loss: 0.7582
Epoch [80/100], Loss: 0.4958
Epoch [90/100], Loss: 0.3256
Epoch [100/100], Loss: 0.2152


In [12]:
# 不用
import numpy as np

# 生成示例数据
X = np.random.randn(100, 1)
y = 3 * X + 2 + np.random.randn(100, 1) * 0.1  # y = 3x + 2 + 噪声

# 初始化参数
w = np.random.randn(1, 1)
b = np.random.randn(1)
learning_rate = 0.01

# 训练模型
for epoch in range(100):
    # 前向传播
    y_pred = np.dot(X, w) + b
    loss = np.mean((y_pred - y) ** 2)  # 均方误差

    # 计算梯度
    dw = (2 / X.shape[0]) * np.dot(X.T, (y_pred - y))
    db = (2 / X.shape[0]) * np.sum(y_pred - y)

    # 更新参数
    w -= learning_rate * dw
    b -= learning_rate * db

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss:.4f}')


Epoch [10/100], Loss: 13.0915
Epoch [20/100], Loss: 8.7977
Epoch [30/100], Loss: 5.9145
Epoch [40/100], Loss: 3.9780
Epoch [50/100], Loss: 2.6771
Epoch [60/100], Loss: 1.8030
Epoch [70/100], Loss: 1.2155
Epoch [80/100], Loss: 0.8206
Epoch [90/100], Loss: 0.5550
Epoch [100/100], Loss: 0.3764


In [13]:
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

In [14]:
print(f"真实参数: w = {true_w}, b = {true_b}")
print(f"学到的参数: w = {model.linear.weight.data}, b = {model.linear.bias.data}")

真实参数: w = tensor([ 2.0000, -3.4000]), b = 4.2
学到的参数: w = tensor([[2.6582]]), b = tensor([1.7459])


In [19]:
import os
os.getcwd()

'D:\\Jupyter'

In [21]:
os.chdir('F:/PythonTest/20250806/')